# Exploring Quantum Horizon — arXiv:2606.14484
**ArXivist-generated exploratory notebook**

This notebook visualizes the paper's five figures: the mining-competitiveness
curve, the bimodal break-year forecast, the Bitcoin exposure decomposition,
the migration-race comparison, and the top-20 readiness ratings. All plots
use this repo's calibrated models -- see `reproduce_arxiv_2606_14484.ipynb`
and `README.md` for what's paper-stated vs. tuned/assumed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_horizon.mining import MiningCompetitivenessModel
from quantum_horizon.timeline import PhysicsBasedEstimator, SurveyBasedEstimator, SystemicForecastModel
from quantum_horizon.exposure import BitcoinExposureModel
from quantum_horizon.migration import MigrationRaceModel
from quantum_horizon.survey import QuantumReadinessSurvey
from quantum_horizon.utils import (
    plot_figure1_hashrate, plot_figure2_forecast, plot_figure3_exposure_pie,
    plot_figure4_migration_race, plot_figure5_readiness,
)

plt.rcParams["figure.figsize"] = (8, 6)


## Figure 1: Effective quantum hashrate vs gate speed

A single quantum machine, even at an optimistic 100 GHz, reaches only
~21 TH/s -- about one-tenth of a single modern ASIC, and utterly dwarfed
by the ~860 EH/s 2026 Bitcoin network.

In [ ]:
model = MiningCompetitivenessModel()
gate_speeds = np.logspace(np.log10(0.0667), np.log10(100), 200)
hashrates = model.effective_hashrate(gate_speeds)

fig = plot_figure1_hashrate(gate_speeds, hashrates, asic_hashrate_ths=200, network_hashrate_ehs=860)
plt.show()


## Figure 2: The bimodal break-year forecast

Two estimators disagree by more than a decade -- an expert-survey hump
peaking in the late 2030s, and a physics/hardware-scaling hump peaking in
the early 2050s. The combined distribution's median falls in the
low-probability trough between them.

In [ ]:
physics = PhysicsBasedEstimator()
survey = SurveyBasedEstimator()
forecast = SystemicForecastModel(physics, survey)
result = forecast.run(n_draws=200000, survey_weight=0.5, seed=0)

fig = plot_figure2_forecast(result["survey_samples"], result["physics_samples"], result["combined_samples"])
plt.show()

print(f"Combined median: {result['median']:.1f} (paper: the median 'falls in the low-probability trough between them')")


## Sensitivity: how much does the survey-vs-physics weight matter?

The paper notes: "at survey weights from 0.25 to 0.75 the by-2035
probability runs from about 8% to 24%... so the near-term tail tracks the
weighting rather than being a hard result."


In [ ]:
sweep_df = forecast.sensitivity_sweep([0.25, 0.35, 0.5, 0.65, 0.75], n_draws=100000, seed=0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df["survey_weight"], sweep_df["cdf_2035"] * 100, "o-", color="tab:red", label="P(CRQC by 2035)")
ax.set_xlabel("survey weight")
ax.set_ylabel("P(CRQC by 2035), %")
ax.set_title("Sensitivity of near-term probability to the survey-physics blend weight")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(sweep_df.to_string(index=False))


## Figure 3: Bitcoin's supply split

Of ~19.4M circulating BTC, only ~2.3M (12%) is irreducibly at risk -- the
rest is either migratable (spendable, can move to safety on warning) or
protected until spent (fresh hash-based addresses).

In [ ]:
btc = BitcoinExposureModel()
decomposition = btc.decompose(total_supply_btc_millions=19.4, irreducible_btc_millions=2.3, migratable_btc_millions=3.7)

fig = plot_figure3_exposure_pie(decomposition)
plt.show()


## Figure 4: The migration race

A prompt 2026 start finishes migration around 2030 -- comfortably ahead of
even the most aggressive 2035 CRQC estimate. Only a severely delayed start
(2033) running against the most aggressive CRQC estimate ends up at risk.

In [ ]:
mr = MigrationRaceModel()
scenarios = {
    "prompt": {"start_year": 2026, "migration_duration_years": 4},
    "delayed": {"start_year": 2030, "migration_duration_years": 5},
    "severe_delay": {"start_year": 2033, "migration_duration_years": 5},
}
crqc_estimates = {"aggressive": 2035, "survey_median": 2040, "conservative": 2050}
df = mr.run_scenarios(scenarios, crqc_estimates)

fig = plot_figure4_migration_race(df, crqc_estimates)
plt.show()

at_risk = df[df["at_risk"]]
print(f"At-risk scenarios: {len(at_risk)}/{len(df)}")
print(at_risk[["scenario", "crqc_estimate_name"]].to_string(index=False) if len(at_risk) else "None")


## Figure 5: Top-20 quantum-readiness ratings

None of the top-20 cryptocurrencies is fully post-quantum. XRP, Solana, and
Zcash lead with live test deployments; Bitcoin and Dogecoin sit near the
bottom.

In [ ]:
survey = QuantumReadinessSurvey()
ratings = survey.load_ratings("../data/table3_readiness_ratings.csv")

fig = plot_figure5_readiness(ratings)
plt.show()

stats = survey.summary_stats(ratings)
print(f"Mean rating: {stats['mean_rating']:.2f}, Median: {stats['median_rating']:.2f}, Max: {stats['max_rating']:.1f}")


## Next steps

- Run `python run_all.py --config configs/config.yaml --output-dir results/`
  for the full end-to-end reproduction with saved Figures 1-5.
- Try tuning `configs/config.yaml`'s timeline parameters
  (`survey_estimator_mode_year`, `physics_calibration_2026_physical_qubits_required`)
  to see how sensitive the combined forecast is to these ASSUMED calibration choices.
- Extend `data/table3_readiness_ratings.csv` to the full ~19-20 coin set the
  paper surveys, if you want a more complete Figure 5.
